# 02. SOVIGAZ DATA PIPELINE & FEATURE ENGINEERING (PHASE 2)
**Project**: SOVIGAZ Machine Learning System  
**Author**: Senior Machine Learning Engineer & Data Scientist  
**Purpose**: Khử trùng lặp, chuẩn hóa dữ liệu, liên kết hình sao chống bùng nổ dòng, và xây dựng các ma trận đặc trưng chuỗi thời gian, RFM và Churn Panel đa thời điểm.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Set project root in path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
sys.path.insert(0, str(project_root))

from config.config import PROCESSED_DATA_DIR, REPORTS_DIR
print(f"Processed Data Dir: {PROCESSED_DATA_DIR}")


## 1. Kiểm tra các Dataset đã xử lý (Processed Parquet)

In [ ]:
files = [
    "processed_sales.parquet",
    "sales_daily.parquet",
    "sales_weekly.parquet",
    "sales_monthly.parquet",
    "customer_master.parquet",
    "rfm_features.parquet",
    "churn_dataset.parquet",
    "customer_credit_snapshot_20260731.parquet"
]

for f in files:
    fp = PROCESSED_DATA_DIR / f
    if fp.exists():
        df_p = pd.read_parquet(fp)
        size_mb = os.path.getsize(fp) / (1024 * 1024)
        print(f"✅ {f:42s}: {len(df_p):10,d} rows | {df_p.shape[1]:2d} cols | {size_mb:6.2f} MB")
    else:
        print(f"❌ {f:42s}: NOT FOUND")


## 2. Kiểm tra Churn Dataset & Phân phối nhãn đa thời điểm

In [ ]:
df_churn = pd.read_parquet(PROCESSED_DATA_DIR / "churn_dataset.parquet")
print(f"Total Churn Samples: {len(df_churn):,}")
display(df_churn.groupby("snapshot_date").agg(
    Total_Customers=("AccountObjectCode", "count"),
    Churned_Y1=("churn_target_90d", "sum"),
    Churn_Rate_Pct=("churn_target_90d", lambda s: round(s.mean() * 100, 2))
).reset_index())
